# Environment Setup

This notebook verifies your OpenShift environment, deploys models on RHOAI, and tests InferenceService endpoints.

## 1. Verify Cluster Access

In [ ]:
import subprocess
import json

print("=" * 50)
print("OpenShift Environment Verification")
print("=" * 50)

checks = [
    ("oc CLI", ["oc", "version", "--client", "-o", "json"]),
    ("Cluster login", ["oc", "whoami"]),
    ("Cluster URL", ["oc", "whoami", "--show-server"]),
]

for name, cmd in checks:
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
        output = result.stdout.strip()
        if result.returncode == 0:
            print(f"✅ {name}: {output[:80]}")
        else:
            print(f"❌ {name}: {result.stderr.strip()[:80]}")
    except FileNotFoundError:
        print(f"❌ {name}: command not found")

## 2. Verify RHOAI Operator

In [ ]:
%%bash
echo "=== RHOAI Operator ==="
oc get csv -n redhat-ods-operator 2>/dev/null | grep -i rhods || echo "⚠️  RHOAI operator not found"

echo ""
echo "=== GPU Nodes ==="
oc get nodes -l nvidia.com/gpu.present=true --no-headers 2>/dev/null || echo "⚠️  No GPU nodes labeled"

echo ""
echo "=== KServe/ModelMesh ==="
oc get crd inferenceservices.serving.kserve.io --no-headers 2>/dev/null && echo "✅ KServe CRD available" || echo "❌ KServe not installed"

## 3. Configure Environment Variables

In [ ]:
from pathlib import Path

env_path = Path("../.env")
sample_path = Path("../sample.env")

if not env_path.exists():
    if sample_path.exists():
        env_path.write_text(sample_path.read_text())
        print(f"Created {env_path} from sample.env")
        print("⚠️  Edit .env and fill in your tokens before proceeding.")
    else:
        print("❌ sample.env not found.")
else:
    print(f"✅ {env_path} already exists.")

## 4. Deploy or Select a Model

This lab requires an OpenAI-compatible model endpoint. Two options:

**Option A (Default)** — Deploy a Qwen model (FP8) in this cluster using the RHOAI 3.4 Inference Server:

| Model | VRAM | Notes |
|-------|------|-------|
| `qwen35-35b-a3b` | ~20 GB | **Default.** MoE (3B active), tool-calling native, RHOAI 3.4 validated |
| `qwen-coder-7b` | ~8 GB | Fastest to deploy, code-focused (Qwen2.5-Coder) |
| `qwen-coder-14b` | ~16 GB | Better code quality (Qwen2.5-Coder) |
| `qwen3-coder-30b` | ~24 GB | Qwen3-Coder MoE |

**Option B** — Use an existing model already deployed in any namespace (e.g. `gemma4` in another project). Set `USE_EXISTING = True` below and fill in the endpoint info.

In [ ]:
import subprocess, json, os
from dotenv import load_dotenv

load_dotenv("../.env")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 📌 Configuration — edit this section to match your environment
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Option A: Deploy a new Qwen model (FP8)
DEPLOY_MODEL = "qwen35-35b-a3b"  # "qwen35-35b-a3b" | "qwen-coder-7b" | "qwen-coder-14b" | "qwen3-coder-30b"

# Option B: Use an existing model already deployed on the cluster
USE_EXISTING = False  # Set True to skip deployment and use existing model
EXISTING_NAMESPACE = "my-models"        # namespace where existing model lives
EXISTING_ISVC_NAME = "gemma4"           # InferenceService name
EXISTING_MODEL_NAME = "gemma-3-4b-it"   # model name for API calls (served-model-name)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def get_isvc_info(name, namespace):
    """Get URL and readiness of an InferenceService."""
    r = subprocess.run(
        ["oc", "get", "inferenceservice", name, "-n", namespace, "-o", "json"],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        return None, None
    data = json.loads(r.stdout)
    url = data.get("status", {}).get("url", "")
    conditions = data.get("status", {}).get("conditions", [])
    ready = next((c["status"] for c in conditions if c["type"] == "Ready"), "Unknown")
    return url, ready

def get_deployed_models(namespace):
    """Return list of InferenceService names in a namespace."""
    r = subprocess.run(
        ["oc", "get", "inferenceservice", "-n", namespace, "-o", "json"],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        return []
    return [item["metadata"]["name"] for item in json.loads(r.stdout).get("items", [])]

# --- Main logic ---

if USE_EXISTING:
    # Option B: Use existing model
    url, ready = get_isvc_info(EXISTING_ISVC_NAME, EXISTING_NAMESPACE)
    if url:
        MODEL_NAME = EXISTING_MODEL_NAME
        MODEL_URL = url
        MODEL_NAMESPACE = EXISTING_NAMESPACE
        print(f"✅ Using existing model: {EXISTING_ISVC_NAME} in namespace '{EXISTING_NAMESPACE}'")
        print(f"   URL:   {url}")
        print(f"   Model: {MODEL_NAME}")
        print(f"   Ready: {ready}")
    else:
        print(f"❌ InferenceService '{EXISTING_ISVC_NAME}' not found in namespace '{EXISTING_NAMESPACE}'.")
        print(f"   Check with: oc get inferenceservice -n {EXISTING_NAMESPACE}")
        MODEL_NAME = MODEL_URL = MODEL_NAMESPACE = None
else:
    # Option A: Deploy Qwen model
    NAMESPACE = "rhoai-models"
    MODEL_MANIFEST_MAP = {
        "qwen35-35b-a3b":  "manifests/04-model-qwen35-35b-a3b.yaml",
        "qwen-coder-7b":   "manifests/01-model-qwen-coder-7b.yaml",
        "qwen-coder-14b":  "manifests/02-model-qwen-coder-14b.yaml",
        "qwen3-coder-30b": "manifests/03-model-qwen3-coder-30b.yaml",
    }
    deployed = get_deployed_models(NAMESPACE)

    if deployed:
        print(f"✅ Models already deployed in '{NAMESPACE}':")
        for name in deployed:
            url, ready = get_isvc_info(name, NAMESPACE)
            print(f"   • {name}  (Ready={ready})")
        print("\n⏭️  Skipping deployment.")
        print("   To redeploy: oc delete inferenceservice --all -n rhoai-models")
    else:
        print(f"No models in '{NAMESPACE}'. Deploying {DEPLOY_MODEL} (FP8)...")
        print("")
        # Apply shared resources (namespace, secret, runtime)
        subprocess.run(["oc", "apply", "-f", "manifests/00-rhoai-models.yaml"],
                       capture_output=True, text=True)
        # Update HF token secret with actual value from .env
        hf_token = os.getenv("HF_TOKEN", "")
        if hf_token and not hf_token.startswith("REPLACE"):
            subprocess.run(
                ["oc", "create", "secret", "generic", "hf-token",
                 f"--from-literal=token={hf_token}", "-n", NAMESPACE,
                 "--dry-run=client", "-o", "yaml"],
                capture_output=True, text=True
            ).stdout
            pipe = subprocess.run(
                ["oc", "create", "secret", "generic", "hf-token",
                 f"--from-literal=token={hf_token}", "-n", NAMESPACE,
                 "--dry-run=client", "-o", "yaml"],
                capture_output=True, text=True)
            subprocess.run(["oc", "apply", "-f", "-"], input=pipe.stdout,
                           capture_output=True, text=True)
        # Apply only the selected model manifest
        model_manifest = MODEL_MANIFEST_MAP[DEPLOY_MODEL]
        r = subprocess.run(["oc", "apply", "-f", model_manifest],
                           capture_output=True, text=True)
        print(r.stdout)
        print(f"✅ Deployed {DEPLOY_MODEL}")
        print("⏳ Downloading weights from HuggingFace (3-10 min on first run).")
        print("   Monitor: oc get pods -n rhoai-models -w")

    # Resolve which model to use going forward
    url, ready = get_isvc_info(DEPLOY_MODEL, NAMESPACE)
    MODEL_NAME = DEPLOY_MODEL
    MODEL_URL = url or f"(pending — wait for {DEPLOY_MODEL} to become Ready)"
    MODEL_NAMESPACE = NAMESPACE

print("")
print("=" * 60)
print(f"  LAB MODEL CONFIG (used in subsequent notebooks):")
print(f"    MODEL_NAME:      {MODEL_NAME}")
print(f"    MODEL_URL:       {MODEL_URL}")
print(f"    MODEL_NAMESPACE: {MODEL_NAMESPACE}")
print("=" * 60)

In [ ]:
import subprocess

ns = MODEL_NAMESPACE or "rhoai-models"
print(f"=== InferenceService in '{ns}' ===")
r = subprocess.run(["oc", "get", "inferenceservice", "-n", ns], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else "(not found)")

print(f"\n=== Pods in '{ns}' ===")
r = subprocess.run(["oc", "get", "pods", "-n", ns], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else "(no pods)")

print("\nTip: Re-run this cell until READY=True, then proceed to Step 5.")

## 5. Test Model Endpoints

Once InferenceServices show `READY=True`, test the endpoints.

In [ ]:
import subprocess, json

ns = MODEL_NAMESPACE or "rhoai-models"

# Get SA token for authentication
token_result = subprocess.run(
    ["oc", "create", "token", "default", "-n", ns, "--duration=1h"],
    capture_output=True, text=True
)
SA_TOKEN = token_result.stdout.strip()

if not MODEL_URL or MODEL_URL.startswith("(pending"):
    print(f"⚠️  Model URL not resolved yet. Wait for deployment to complete.")
    print(f"   Re-run Step 4 after the model is Ready.")
else:
    print(f"Testing endpoint: {MODEL_URL}/v1/models")
    print(f"Model: {MODEL_NAME}")
    print("=" * 60)

    test_result = subprocess.run(
        ["curl", "-sk", f"{MODEL_URL}/v1/models",
         "-H", f"Authorization: Bearer {SA_TOKEN}"],
        capture_output=True, text=True, timeout=15
    )

    if test_result.returncode == 0 and test_result.stdout.strip():
        try:
            models = json.loads(test_result.stdout)
            print(f"✅ Responding! Available models: {[m['id'] for m in models.get('data', [])]}")
        except json.JSONDecodeError:
            print(f"⚠️  Got response but not valid JSON:\n{test_result.stdout[:200]}")
    else:
        print(f"❌ Not responding yet. Check pod status:")
        print(f"   oc get pods -n {ns}")

In [ ]:
import subprocess, json

ns = MODEL_NAMESPACE or "rhoai-models"
token_r = subprocess.run(
    ["oc", "create", "token", "default", "-n", ns, "--duration=1h"],
    capture_output=True, text=True
)
token = token_r.stdout.strip()

if not MODEL_URL or MODEL_URL.startswith("(pending"):
    print("⚠️  Model not ready. Re-run Step 4 status check.")
else:
    print(f"Inference test → {MODEL_NAME}")
    print("-" * 50)
    r = subprocess.run(
        ["curl", "-sk", f"{MODEL_URL}/v1/chat/completions",
         "-H", f"Authorization: Bearer {token}",
         "-H", "Content-Type: application/json",
         "-d", json.dumps({
             "model": MODEL_NAME,
             "messages": [{"role": "user", "content": "Write a Python hello world one-liner."}],
             "max_tokens": 30
         })],
        capture_output=True, text=True, timeout=30
    )
    if r.returncode == 0 and r.stdout.strip():
        resp = json.loads(r.stdout)
        content = resp["choices"][0]["message"]["content"]
        print(f"✅ Response:\n{content}")
    else:
        print(f"❌ No response. stderr: {r.stderr[:200]}")

## 6. Verify MaaS Availability

Check if MaaS (Models as a Service) is available on your cluster.

In [ ]:
%%bash
echo "=== MaaS Gateway ==="
oc get gateway -n openshift-ingress maas-default-gateway 2>/dev/null && echo "✅ MaaS Gateway found" || echo "⚠️  MaaS Gateway not found — install MaaS via RHOAI operator"

echo ""
echo "=== MaaS Endpoint ==="
CLUSTER_DOMAIN=$(oc get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}' 2>/dev/null)
if [ -n "$CLUSTER_DOMAIN" ]; then
    echo "  https://maas.${CLUSTER_DOMAIN}"
else
    echo "⚠️  Could not determine cluster domain"
fi

## Next Steps

Once models are ready (`oc get inferenceservice -n rhoai-models` shows `READY=True`):

1. **Phase 1** → `1_mcp_servers/2_deploy_mcp_servers.ipynb` to deploy MCP tool servers
2. **Phase 2** → `2_ai_gateway/2_enable_maas.ipynb` to enable Models as a Service